# Inference

Loads the registered pipeline from MLflow and runs it on the raw test set. The pipeline does its own preprocessing, so we feed in raw DataFrame rows.

## 1. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
for p in ['.', '..', '/kaggle/working/ML_Asgn2']:
    if os.path.isdir(os.path.join(p, 'src')) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from src.data_utils import load_test
from src.mlflow_utils import init_tracking

init_tracking('Inference')

## 2. Load registered pipeline

In [ ]:
MODEL_NAME = 'IEEEFraudBestModel'
MODEL_VERSION = 'latest'   # or pin to a specific integer version like '3'

model_uri = f'models:/{MODEL_NAME}/{MODEL_VERSION}'
pipeline = mlflow.sklearn.load_model(model_uri)
print(f'Loaded {model_uri}')
print(f'Pipeline steps: {[name for name, _ in pipeline.steps]}')

## 3. Load raw test set

In [ ]:
X_test, transaction_ids = load_test()
print(f'test rows: {len(X_test):,}, columns: {X_test.shape[1]}')

## 4. Predict and build submission

In [ ]:
# Pipeline preprocesses internally, so we feed it the raw frame directly.
fraud_proba = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud': fraud_proba,
})
os.makedirs('submissions', exist_ok=True)
submission_path = 'submissions/submission.csv'
submission.to_csv(submission_path, index=False)

print(f'wrote {submission_path}  ({len(submission):,} rows)')
print(submission.head())